<a href="https://colab.research.google.com/github/MSagri05/IAT461-Final-Project/blob/main/workplace_accommodation_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workplace Accommodation Bias and Fulfillment Analysis

**Course:** IAT 461  
**Final Project – Data Scientist Role**  
**Client:** Stin  
**Data Scientist:** Manmeet Sagri  

## Project Overview

This project analyzes a synthetic workplace accommodation dataset created for the Job Accommodation Network (JAN). The purpose of the initial exploratory data analysis is to evaluate three possible organizational strategies and determine whether accommodation outcomes are primarily influenced by cost or whether specific disability categories, departments, and HR reviewers experience disproportionate denial rates or implementation delays.

The later modeling stage will predict accommodation approval status and implementation time while accounting for relevant employee, request, workplace, and cost-related variables.

### importing the dataset from google drive

In [5]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Environment Setup


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Data Loading

dataset is stored in google drive, so it can accessed even if runtime is disconnected

In [7]:
file_path = (
    "/content/drive/MyDrive/IAT461_Final_Project/"
    "data/workplace_accommodation_synthetic.csv"
)

try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully.")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
except FileNotFoundError:
    print("File not found. Check the folder and file names in Google Drive.")

Dataset loaded successfully.
Rows: 260
Columns: 16


In [8]:
df.head()

,request_id,disability_category,accommodation_category,employee_tenure_years,hourly_wage_usd,has_college_degree,is_cost_incurred,accommodation_cost_usd,request_severity_level,remote_work_status,department,hr_reviewer_id,approval_status,denial_reason,days_to_implement,retained_6mo
0,REQ-1000,Mental Health,Assistive technology,4.3,22.46,1,0,0.00,Medium,Hybrid,Sales,HR-107,Denied,Undue Hardship,NaN,0
1,REQ-1001,Chronic Medical,Assistive technology,5.6,21.83,1,0,0.00,Low,Remote,Operations,HR-101,Denied,Alternative Offered,NaN,0
2,REQ-1002,Cognitive/Neurological,Communication aid,7.5,26.43,1,1,2321.51,Medium,Hybrid,Engineering,HR-103,Approved,NaN,16.0,1
3,REQ-1003,Chronic Medical,Schedule modification,1.6,20.24,1,0,0.00,Low,On-site,Operations,HR-104,Denied,Undue Hardship,NaN,0
4,REQ-1004,Chronic Medical,Schedule modification,2.3,25.84,0,1,79.80,Low,Hybrid,Operations,HR-104,Approved,NaN,19.0,1


In [9]:
df.tail()

,request_id,disability_category,accommodation_category,employee_tenure_years,hourly_wage_usd,has_college_degree,is_cost_incurred,accommodation_cost_usd,request_severity_level,remote_work_status,department,hr_reviewer_id,approval_status,denial_reason,days_to_implement,retained_6mo
255,REQ-1255,Cognitive/Neurological,Assistive technology,1.4,27.95,1,0,0.0,Low,Hybrid,Engineering,HR-107,Denied,Lack of Documentation,NaN,1
256,REQ-1256,Physical/Mobility,Schedule modification,19.4,19.92,1,0,0.0,Medium,Hybrid,Finance,HR-107,Approved,NaN,7.0,1
257,REQ-1257,Chronic Medical,Assistive technology,2.4,22.72,0,0,0.0,High,On-site,Finance,HR-103,Approved,NaN,32.0,1
258,REQ-1258,Sensory,Assistive technology,3.4,16.31,1,0,0.0,Low,On-site,Customer Support,HR-106,Approved,NaN,5.0,1
259,REQ-1259,Sensory,Policy modification,2.4,25.65,1,0,0.0,Low,On-site,Customer Support,HR-108,Approved,NaN,2.0,1


In [10]:
df.sample(5, random_state=42)

,request_id,disability_category,accommodation_category,employee_tenure_years,hourly_wage_usd,has_college_degree,is_cost_incurred,accommodation_cost_usd,request_severity_level,remote_work_status,department,hr_reviewer_id,approval_status,denial_reason,days_to_implement,retained_6mo
30,REQ-1030,Cognitive/Neurological,Assistive technology,1.9,33.68,1,0,0.00,Low,Hybrid,Operations,HR-110,Partially Approved,NaN,3.0,1
181,REQ-1181,Cognitive/Neurological,Schedule modification,8.1,27.71,1,0,0.00,Medium,On-site,Customer Support,HR-102,Approved,NaN,49.0,1
223,REQ-1223,Mental Health,Policy modification,5.0,18.26,0,1,95.48,Low,Remote,Sales,HR-102,Denied,Lack of Documentation,NaN,1
185,REQ-1185,Mental Health,Physical workspace,3.3,22.43,1,1,1322.50,Medium,On-site,Engineering,HR-103,Denied,Undue Hardship,NaN,1
211,REQ-1211,Mental Health,Schedule modification,3.9,21.87,0,0,0.00,Low,On-site,Sales,HR-107,Partially Approved,NaN,31.0,1


## 3. Dataset Overview

This section provides a high-level overview of the dataset, including its size, column names, data types, summary statistics, and unique values. This helps identify the structure of the data before conducting deeper analysis.

In [11]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Dataset shape: (260, 16)

Column names:
['request_id', 'disability_category', 'accommodation_category', 'employee_tenure_years', 'hourly_wage_usd', 'has_college_degree', 'is_cost_incurred', 'accommodation_cost_usd', 'request_severity_level', 'remote_work_status', 'department', 'hr_reviewer_id', 'approval_status', 'denial_reason', 'days_to_implement', 'retained_6mo']


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260 entries, 0 to 259
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   request_id              260 non-null    object 
 1   disability_category     260 non-null    object 
 2   accommodation_category  260 non-null    object 
 3   employee_tenure_years   260 non-null    float64
 4   hourly_wage_usd         260 non-null    float64
 5   has_college_degree      260 non-null    int64  
 6   is_cost_incurred        260 non-null    int64  
 7   accommodation_cost_usd  260 non-null    float64
 8   request_severity_level  260 non-null    object 
 9   remote_work_status      260 non-null    object 
 10  department              260 non-null    object 
 11  hr_reviewer_id          260 non-null    object 
 12  approval_status         260 non-null    object 
 13  denial_reason           69 non-null     object 
 14  days_to_implement       191 non-null    fl

In [13]:
df.describe()

,employee_tenure_years,hourly_wage_usd,has_college_degree,is_cost_incurred,accommodation_cost_usd,days_to_implement,retained_6mo
count,260.000000,260.000000,260.000000,260.000000,260.000000,191.000000,260.000000
mean,5.460769,23.103385,0.684615,0.380769,260.160769,18.382199,0.784615
std,3.988177,5.340462,0.465565,0.486512,645.648811,13.469478,0.411882
min,0.400000,9.450000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,2.400000,19.562500,0.000000,0.000000,0.000000,9.000000,1.000000
50%,4.500000,22.630000,1.000000,0.000000,0.000000,15.000000,1.000000
75%,7.600000,26.662500,1.000000,1.000000,210.995000,25.000000,1.000000
max,20.800000,38.340000,1.000000,1.000000,7588.770000,72.000000,1.000000


In [14]:
column_summary = pd.DataFrame({
    "data_type": df.dtypes,
    "non_null_count": df.notnull().sum(),
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df) * 100).round(2),
    "unique_values": df.nunique()
})

column_summary

,data_type,non_null_count,missing_count,missing_percentage,unique_values
request_id,object,260,0,0.00,260
disability_category,object,260,0,0.00,5
accommodation_category,object,260,0,0.00,5
employee_tenure_years,float64,260,0,0.00,113
hourly_wage_usd,float64,260,0,0.00,244
has_college_degree,int64,260,0,0.00,2
is_cost_incurred,int64,260,0,0.00,2
accommodation_cost_usd,float64,260,0,0.00,100
request_severity_level,object,260,0,0.00,3
remote_work_status,object,260,0,0.00,3


### Duplicate Records

Duplicate records can distort approval rates, averages, and model results. The dataset is checked for repeated rows and repeated request identifiers.

In [15]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated request IDs:", df["request_id"].duplicated().sum())

Fully duplicated rows: 0
Duplicated request IDs: 0


### Categorical Value Review

The unique values in each categorical column are reviewed to identify inconsistent labels, spelling differences, or unexpected categories.

In [16]:
categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))


request_id
request_id
REQ-1259    1
REQ-1000    1
REQ-1001    1
REQ-1002    1
REQ-1243    1
           ..
REQ-1008    1
REQ-1007    1
REQ-1006    1
REQ-1005    1
REQ-1004    1
Name: count, Length: 260, dtype: int64

disability_category
disability_category
Mental Health             64
Physical/Mobility         59
Cognitive/Neurological    53
Sensory                   44
Chronic Medical           40
Name: count, dtype: int64

accommodation_category
accommodation_category
Schedule modification    75
Assistive technology     70
Physical workspace       58
Policy modification      30
Communication aid        27
Name: count, dtype: int64

request_severity_level
request_severity_level
Low       112
Medium    110
High       38
Name: count, dtype: int64

remote_work_status
remote_work_status
On-site    122
Hybrid     103
Remote      35
Name: count, dtype: int64

department
department
Customer Support    72
Operations          61
Sales               53
Engineering         42
Finance            

## 4. Missing Value Analysis

Missing values are examined before cleaning the dataset. In this dataset, some missing values are expected because they depend on the outcome of the accommodation request. For example, denied requests do not have an implementation time, while approved requests generally do not have a denial reason. Therefore, these missing values should not automatically be treated as data errors or filled using imputation.

In [17]:
missing_values = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_values[missing_values["missing_count"] > 0].sort_values(
    by="missing_percentage",
    ascending=False
)

,missing_count,missing_percentage
denial_reason,191,73.46
days_to_implement,69,26.54


In [18]:
missing_days_by_status = pd.crosstab(
    df["approval_status"],
    df["days_to_implement"].isna(),
    margins=True
)

missing_days_by_status.columns = ["days_available", "days_missing", "total"]
missing_days_by_status

,days_available,days_missing,total
approval_status,,,
Approved,141,0,141
Denied,0,69,69
Partially Approved,50,0,50
All,191,69,260


In [19]:
missing_reason_by_status = pd.crosstab(
    df["approval_status"],
    df["denial_reason"].isna(),
    margins=True
)

missing_reason_by_status.columns = ["reason_available", "reason_missing", "total"]
missing_reason_by_status

,reason_available,reason_missing,total
approval_status,,,
Approved,0,141,141
Denied,69,0,69
Partially Approved,0,50,50
All,69,191,260


### Missing Value Interpretation

The missing values appear to be structurally related to the request outcome rather than randomly missing. Requests that were denied generally do not have a value for `days_to_implement` because the accommodation was never implemented. Similarly, approved requests generally do not have a `denial_reason`.

For this reason, these missing values will not be filled using the mean, median, or mode. Instead, they will be handled according to the analysis being performed. For example, implementation-delay analysis will only use requests with a recorded implementation time.